## 🔢 원본 데이터 로드 및 전체 파이프라인 실행

이제 학습된 모델을 사용하여 원본 train.csv 데이터에 대해 문단별 점수를 계산하고 문서 라벨을 예측합니다.

In [ ]:
# 원본 데이터 로드
TRAIN_DATA_PATH = '/Users/youngjinson/Downloads/open/train.csv'
df_train = pd.read_csv(TRAIN_DATA_PATH)

print(f"📊 원본 학습 데이터 로드 완료")
print(f"  - 총 문서 수: {len(df_train):,}개")
print(f"  - 컬럼: {list(df_train.columns)}")
print(f"\n레이블 분포:")
print(df_train['generated'].value_counts())

df_train.head()

In [ ]:
# 전체 문서 텍스트 추출
all_doc_texts = df_train['full_text'].tolist()
all_doc_labels = df_train['generated'].tolist()

print(f"전체 문서: {len(all_doc_texts):,}개")
print(f"  - Human (0): {sum([1 for l in all_doc_labels if l == 0]):,}개")
print(f"  - AI (1): {sum([1 for l in all_doc_labels if l == 1]):,}개")

### 전체 문서 예측 (Best 모델 사용)

학습된 best 모델을 로드하여 전체 문서에 대해 예측합니다.

In [ ]:
# Best 모델 로드
checkpoint = torch.load(f'{checkpoint_dir}/best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Best 모델 로드 완료 (F1: {checkpoint['f1']:.4f})")

In [ ]:
# 전체 문서 예측 (샘플링하여 테스트)
# 전체를 한번에 처리하면 시간이 오래 걸리므로 먼저 샘플로 테스트
SAMPLE_SIZE = 1000  # 샘플 크기 (전체를 처리하려면 len(all_doc_texts)로 변경)

sample_indices = np.random.choice(len(all_doc_texts), SAMPLE_SIZE, replace=False)
sample_texts = [all_doc_texts[i] for i in sample_indices]
sample_labels = [all_doc_labels[i] for i in sample_indices]

print(f"샘플 데이터: {SAMPLE_SIZE:,}개 문서")
print(f"  - Human: {sum([1 for l in sample_labels if l == 0]):,}개")
print(f"  - AI: {sum([1 for l in sample_labels if l == 1]):,}개")

In [ ]:
# 문서 예측 실행
print("\n🔮 문서 예측 시작...\n")
doc_predictions, paragraph_predictions = predict_documents(
    model, sample_texts, tokenizer, device, threshold=0.5, use_weighted=False
)

# 실제 레이블 추가
doc_predictions['true_label'] = sample_labels

print("\n✅ 예측 완료!")
print(f"\n문서별 결과: {len(doc_predictions)}개")
print(f"문단별 점수: {len(paragraph_predictions)}개")

In [ ]:
# 성능 평가
doc_accuracy = accuracy_score(doc_predictions['true_label'], doc_predictions['predicted_label'])
doc_f1 = f1_score(doc_predictions['true_label'], doc_predictions['predicted_label'])
doc_auc = roc_auc_score(doc_predictions['true_label'], doc_predictions['avg_ai_score'])

print(f"\n📈 문서 레벨 성능:")
print(f"  - Accuracy: {doc_accuracy:.4f}")
print(f"  - F1 Score: {doc_f1:.4f}")
print(f"  - ROC AUC: {doc_auc:.4f}")

In [ ]:
# Confusion Matrix
cm_doc = confusion_matrix(doc_predictions['true_label'], doc_predictions['predicted_label'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm_doc, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Human', 'AI'], 
            yticklabels=['Human', 'AI'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'문서 레벨 Confusion Matrix (n={SAMPLE_SIZE})')
plt.savefig(f'{checkpoint_dir}/document_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

### 문단별 점수 분포 시각화

In [ ]:
# 문단별 AI 점수 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 문단 점수 분포
axes[0].hist(paragraph_predictions['ai_score'], bins=50, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('AI Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('문단별 AI 점수 분포')
axes[0].axvline(x=0.5, color='r', linestyle='--', label='Threshold=0.5')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 문서별 평균 AI 점수 분포
axes[1].hist(doc_predictions['avg_ai_score'], bins=50, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Average AI Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('문서별 평균 AI 점수 분포')
axes[1].axvline(x=0.5, color='r', linestyle='--', label='Threshold=0.5')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{checkpoint_dir}/score_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

### 반복 학습 메커니즘 (Iteration 1)

Human으로 예측된 문서의 문단 점수를 0으로 강제 조정하여 재학습합니다.

In [ ]:
# Human으로 예측된 문서의 doc_id 추출
human_predicted_doc_ids = doc_predictions[doc_predictions['predicted_label'] == 0]['doc_id'].tolist()

print(f"Human으로 예측된 문서: {len(human_predicted_doc_ids)}개")

# 해당 문서의 문단 점수를 0으로 강제 조정
paragraph_predictions_adjusted = paragraph_predictions.copy()
mask = paragraph_predictions_adjusted['doc_id'].isin(human_predicted_doc_ids)
paragraph_predictions_adjusted.loc[mask, 'ai_score'] = 0.0

print(f"조정된 문단 수: {mask.sum()}개")
print(f"\n조정 전 평균 AI 점수: {paragraph_predictions['ai_score'].mean():.4f}")
print(f"조정 후 평균 AI 점수: {paragraph_predictions_adjusted['ai_score'].mean():.4f}")

In [ ]:
# 조정된 점수로 문서 재집계
doc_predictions_iter1 = []

for doc_id in doc_predictions['doc_id'].unique():
    doc_para_scores = paragraph_predictions_adjusted[paragraph_predictions_adjusted['doc_id'] == doc_id]['ai_score'].values
    avg_score = np.mean(doc_para_scores)
    predicted_label = 1 if avg_score > 0.5 else 0
    
    doc_predictions_iter1.append({
        'doc_id': doc_id,
        'avg_ai_score_iter1': avg_score,
        'predicted_label_iter1': predicted_label
    })

doc_predictions_iter1_df = pd.DataFrame(doc_predictions_iter1)
doc_predictions_final = doc_predictions.merge(doc_predictions_iter1_df, on='doc_id')

print("\n✅ Iteration 1 완료!")
print(doc_predictions_final.head())

In [ ]:
# Iteration 1 성능 평가
iter1_accuracy = accuracy_score(doc_predictions_final['true_label'], doc_predictions_final['predicted_label_iter1'])
iter1_f1 = f1_score(doc_predictions_final['true_label'], doc_predictions_final['predicted_label_iter1'])
iter1_auc = roc_auc_score(doc_predictions_final['true_label'], doc_predictions_final['avg_ai_score_iter1'])

print(f"\n📈 Iteration 1 성능:")
print(f"  - Accuracy: {iter1_accuracy:.4f} (초기: {doc_accuracy:.4f})")
print(f"  - F1 Score: {iter1_f1:.4f} (초기: {doc_f1:.4f})")
print(f"  - ROC AUC: {iter1_auc:.4f} (초기: {doc_auc:.4f})")

### 결과 저장

In [ ]:
# 최종 결과 저장
doc_predictions_final.to_csv(f'{checkpoint_dir}/document_predictions_final.csv', index=False)
paragraph_predictions_adjusted.to_csv(f'{checkpoint_dir}/paragraph_scores_adjusted.csv', index=False)

# 성능 요약
performance_summary = {
    'initial': {
        'accuracy': doc_accuracy,
        'f1': doc_f1,
        'auc': doc_auc
    },
    'iteration_1': {
        'accuracy': iter1_accuracy,
        'f1': iter1_f1,
        'auc': iter1_auc
    }
}

with open(f'{checkpoint_dir}/performance_summary.json', 'w') as f:
    json.dump(performance_summary, f, indent=2)

print("\n✅ 모든 결과 저장 완료!")
print(f"📁 저장 위치: {checkpoint_dir}")